# DuckDB walkthrough: 20M Runestone events

This notebook is the "what's in the log?" pass. I'm using DuckDB because the
file is ~20 million rows and I don't want to pull all of it into pandas just
to count things.

You need `runestone_event_log.parquet` in the project root. If a cell errors on that,
the rest of the analysis can still use the saved CSVs from `python -m ebook_analysis`.


In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

from ebook_analysis.duckdb_queries import connect, query, run_pre_exam_views

con = connect()
con.execute("SELECT 1").fetchone()


(1,)

First, the obvious checks: how many rows, how many semesters, how far back the log goes.
This is the same query as `sql/01_event_overview.sql`.


In [2]:
overview = query(con, "01_event_overview.sql")
overview.T


,0
n_events,19805383
n_semesters,10
n_students,214
n_problems,9873
n_sessions,21891
first_event,2021-02-11 21:16:02
last_event,2026-02-03 21:36:57.276096


Semester counts are a good "did all the files actually load?" check. F21 through F25
plus the winter terms should show up. If one bar is tiny, that semester's export
probably didn't make it into `runestone_event_log.parquet`.


In [3]:
by_semester = query(con, "02_events_by_semester.sql")
by_semester


,semester,n_events,n_students,n_active_days
0,F21,1285969,114,104
1,F22,2195138,159,154
2,F23,1888245,154,94
3,F24,2227329,156,170
4,F25,7446085,129,161
5,W20,404,14,25
6,W21,5,1,1
7,W22,1333268,145,215
8,W23,2130934,214,126
9,W24,1298006,193,165


Exam windows are not inferred here. `exam_windows/` already has timed start/finish
markers. We just collapse extra sessions into one window per student and exam.


In [ ]:
windows = query(con, "03_exam_windows.sql")
windows.groupby(["semester_raw", "midterm"]).size().rename("n_windows").reset_index().head(20)


`04_pre_exam_events.sql` builds a view of practice *before* each student's exam.
That's the leaky part to get right: if exam clicks sneak into "practice," every
quality feature looks better than it is.


In [ ]:
pre_count = run_pre_exam_views(con)
pre_count


In [ ]:
# peek at a few pre-exam rows without dumping the whole view
con.execute('''
    SELECT semester_raw, midterm, family, days_before, level_chapter, problem_name
    FROM pre_exam_events
    WHERE family = 'parsons'
    LIMIT 8
''').df()


That's enough to trust the SQL layer. Notebook 02 joins these features onto
midterm scores and looks at Parsons.
